# Estudio 06 · Alerta combinada

Preregistro `COMBINED_ALERT` (commit `4d9d97c`, antes de calcular). Alerta combinada = probabilidad calibrada >= 0,50
**o** la alerta de la persistencia del protocolo (`persistencia > umbral_brote_casos` de la semana objetivo y
`persistencia >= 2`). Sobre las predicciones guardadas de d3 (base) y b3 (retadora), sin reentrenar; cada una frente a
sí misma con la alerta solo por p >= 0,50. Regla `decide_combined_alert`: F1 a 0,50 principal sobre 2010, 2015, 2016,
2022, 2023 y 2026; precisión y antelación como guardas; adopción con >= 2 horizontes ganados y sin «pierde» ni
«compensa». Medianas de 5 semillas.

In [1]:
import json
import sys

import pandas as pd

from denguekay.config.paths import MODELS, RESEARCH
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase_2026 as p26
import phase_combined_alert as ca
import preregistration as pr

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
METRICS = STUDY / "docs" / "metricas" / "alerta_combinada.json"
EXP = MODELS / "experimentos" / "mejora_modelo"
PATHS = {"d3": EXP / "matriz_clima" / "predicciones_d3.csv.gz", "b3": EXP / "temporada_2026" / "predicciones_b3.csv.gz"}
inp = p26.inputs_2026()
series = inp["serie"]
outbreaks = p26.outbreak_weeks(series)
runs = {k: {s: ca.evaluate_both(p, series) for s, p in p26.read_saved(path).items()} for k, path in PATHS.items()}
outbreaks

{2: 89, 3: 89, 4: 89}

## 1 · Veredicto (regla preregistrada)

In [2]:
SEED = pr.SEEDS[0]
six = pr.SEASON_2026_DECIDING
med, verdicts, reading, adopted, leads = {}, {}, {}, {}, {}
for k, r in runs.items():
    m = {a: pa.seed_median([e[a]["metricas"] for e in r.values()]) for a in ("sola", "combinada")}
    ranges = pa.tie_threshold(*(pa.seed_ranges([e[a]["metricas"] for e in r.values()]) for a in ("sola", "combinada")))
    lead = pr.lead_time_difference_multi(r[SEED]["combinada"]["antelacion"], r[SEED]["sola"]["antelacion"], "alerta_050", six)
    verdicts[k] = pr.decide_combined_alert(m["combinada"], m["sola"], ranges, lead, outbreaks)
    adopted[k] = pr.adopt_combined_alert(verdicts[k])
    blocks = [pr.block_name(s) for s in six]
    sel = lambda x: x[x.bloque.isin(blocks)]
    reading[k] = pr.decide_multi_season(sel(m["combinada"]), sel(m["sola"]), sel(ranges), lead, seasons=six,
                                        min_in_favour=pr.SEASON_2026_MIN_IN_FAVOUR, guard_veto=pr.SEASON_2026_GUARD_VETO)
    med[k], leads[k] = m, lead
for k in runs:
    print(k, "adopta:", adopted[k])
    display(verdicts[k])
    print("lectura con decide_multi_season (la antelación también puede ganar):", reading[k].veredicto.tolist())
leads["d3"].round(3)

d3 adopta: False


,horizonte,veredicto,temporadas,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_precision_050,a_favor_antelacion,en_contra_f1_050,en_contra_precision_050,en_contra_antelacion
0,2,empate,6,precision_050,NaN,3,0,2,1,6,0
1,3,empate,6,precision_050,NaN,4,0,3,2,5,0
2,4,empate,6,precision_050,antelacion,4,0,5,0,5,0


lectura con decide_multi_season (la antelación también puede ganar): ['empate', 'empate', 'gana']
b3 adopta: False


,horizonte,veredicto,temporadas,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_precision_050,a_favor_antelacion,en_contra_f1_050,en_contra_precision_050,en_contra_antelacion
0,2,empate,6,precision_050,NaN,3,0,3,2,6,0
1,3,empate,6,precision_050,NaN,2,0,3,2,5,0
2,4,empate,6,precision_050,antelacion,4,0,5,0,5,0


lectura con decide_multi_season (la antelación también puede ganar): ['empate', 'empate', 'gana']


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,diferencia_temporada_2026,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023,episodios_temporada_2026
0,2,213,0.038,0.014,0.066,0.0,0.029,0.029,0.053,0.053,0.05,30,34,34,38,57,20
1,3,213,0.052,0.023,0.080,0.0,0.029,0.059,0.000,0.053,0.25,30,34,34,38,57,20
2,4,213,0.099,0.061,0.141,0.0,0.088,0.118,0.184,0.070,0.15,30,34,34,38,57,20


## 2 · Precisión, recall y F1 por temporada y horizonte

In [3]:
REP = [pr.block_name(s) for s in pr.COMBINED_ALERT_REPORTED] + ["posterior_2026"]
cols = ["precision_050", "recall_050", "f1_050", "vp_050", "fp_050", "fn_050"]
cells = []
for k in runs:
    for a in ("sola", "combinada"):
        x = med[k][a]
        cells.append(x[x.bloque.isin(REP)][["horizonte", "bloque", *cols]].assign(variante=k, alerta=a))
cells = pd.concat(cells, ignore_index=True)
wide = cells.pivot_table(index=["variante", "bloque", "horizonte"], columns="alerta", values=["precision_050", "recall_050", "f1_050"])
wide.round(3)

f1_050        precision_050        recall_050       
alerta                            combinada   sola     combinada   sola  combinada   sola
variante bloque         horizonte                                                        
b3       posterior_2026 2             0.850  0.813         0.911  0.917      0.796  0.731
                        3             0.796  0.749         0.902  0.914      0.713  0.635
                        4             0.766  0.689         0.902  0.920      0.665  0.551
         temporada_2010 2             0.702  0.709         0.696  0.760      0.705  0.667
                        3             0.637  0.634         0.633  0.788      0.641  0.534
                        4             0.547  0.501         0.547  0.792      0.547  0.368
         temporada_2015 2             0.795  0.771         0.802  0.856      0.788  0.702
                        3             0.747  0.668         0.767  0.851      0.727  0.549
                        4             0.703  0.535         0.728  0.880      0.679  0.384
         temporada_2016 2             0.653  0.653         0.596  0.610      0.722  0.704
                        3             0.620  0.631         0.563  0.617      0.689  0.646
                        4             0.591  0.594         0.533  0.614      0.661  0.572
         temporada_2017 2             0.698  0.699         0.633  0.667      0.772  0.742
                        3             0.654  0.657         0.596  0.653      0.727  0.664
                        4             0.629  0.624         0.582  0.660      0.685  0.592
         temporada_2022 2             0.690  0.700         0.681  0.751      0.701  0.657
                        3             0.685  0.695         0.662  0.771      0.710  0.636
                        4             0.643  0.635         0.615  0.756      0.673  0.548
         temporada_2023 2             0.865  0.861         0.873  0.886      0.856  0.837
                        3             0.840  0.837         0.858  0.889      0.823  0.791
                        4             0.830  0.819         0.861  0.908      0.801  0.746
         temporada_2024 2             0.803  0.805         0.758  0.767      0.853  0.846
                        3             0.792  0.797         0.734  0.750      0.858  0.848
                        4             0.756  0.770         0.697  0.732      0.825  0.810
         temporada_2026 2             0.744  0.671         0.813  0.860      0.685  0.551
                        3             0.754  0.611         0.800  0.800      0.708  0.494
                        4             0.647  0.455         0.692  0.698      0.607  0.337
d3       posterior_2026 2             0.854  0.845         0.912  0.922      0.802  0.784
                        3             0.796  0.779         0.902  0.919      0.713  0.677
                        4             0.770  0.736         0.903  0.926      0.671  0.611
         temporada_2010 2             0.695  0.720         0.698  0.793      0.692  0.658
                        3             0.636  0.602         0.630  0.765      0.641  0.491
                        4             0.547  0.443         0.547  0.745      0.547  0.316
         temporada_2015 2             0.793  0.768         0.801  0.852      0.786  0.702
                        3             0.747  0.665         0.769  0.847      0.727  0.548
                        4             0.703  0.518         0.728  0.864      0.679  0.370
         temporada_2016 2             0.660  0.660         0.594  0.608      0.742  0.724
                        3             0.615  0.622         0.552  0.595      0.694  0.653
                        4             0.585  0.581         0.523  0.587      0.666  0.580
         temporada_2017 2             0.704  0.703         0.651  0.694      0.766  0.714
                        3             0.659  0.658         0.607  0.688      0.720  0.631
                        4             0.629  0.620         0.588  0.697      

## 3 · Falsas alarmas añadidas (años normales y de Niño)

In [4]:
added = {k: ca.added_false_alarms(med[k]["sola"], med[k]["combinada"]) for k in runs}
regime = pd.concat([ca.false_alarms_by_regime(v).assign(variante=k) for k, v in added.items()], ignore_index=True)
regime

,horizonte,fp_anadidas,vp_anadidos,fp_050_sola,regimen,bloques,variante
0,2,163.0,115.0,742.0,normales,"temporada_2010, temporada_2015, temporada_2016...",d3
1,3,290.0,221.0,737.0,normales,"temporada_2010, temporada_2015, temporada_2016...",d3
2,4,470.0,388.0,675.0,normales,"temporada_2010, temporada_2015, temporada_2016...",d3
3,2,60.0,40.0,188.0,nino,"temporada_2017, temporada_2026, posterior_2026",d3
4,3,107.0,71.0,171.0,nino,"temporada_2017, temporada_2026, posterior_2026",d3
5,4,139.0,89.0,153.0,nino,"temporada_2017, temporada_2026, posterior_2026",d3
6,2,158.0,108.0,719.0,normales,"temporada_2010, temporada_2015, temporada_2016...",b3
7,3,313.0,214.0,672.0,normales,"temporada_2010, temporada_2015, temporada_2016...",b3
8,4,497.0,365.0,602.0,normales,"temporada_2010, temporada_2015, temporada_2016...",b3
9,2,51.0,39.0,218.0,nino,"temporada_2017, temporada_2026, posterior_2026",b3


## 4 · HU0007-4 por celda con la alerta combinada

In [5]:
hu = pd.concat([sp.hu0007_4_compliance(med[k]["combinada"][med[k]["combinada"].bloque.isin(REP)]).assign(variante=k)
                for k in runs], ignore_index=True)
print("celdas:", len(hu), "| cumplen clasificación:", int(hu.cumple_clasificacion.sum()),
      "| cumplen regresión:", int(hu.cumple_regresion.sum()))
hu[["variante", "horizonte", "bloque", "precision_050", "recall_050", "f1_050", "cumple_precision_050", "cumple_recall_050",
    "cumple_f1_050", "cumple_clasificacion"]].round(3)

celdas: 54 | cumplen clasificación: 12 | cumplen regresión: 12


,variante,horizonte,bloque,precision_050,recall_050,f1_050,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_clasificacion
0,d3,2,posterior_2026,0.912,0.802,0.854,True,True,True,True
1,d3,2,temporada_2010,0.698,0.692,0.695,True,False,False,False
2,d3,2,temporada_2015,0.801,0.786,0.793,True,False,True,False
3,d3,2,temporada_2016,0.594,0.742,0.660,False,False,False,False
4,d3,2,temporada_2017,0.651,0.766,0.704,True,False,True,False
5,d3,2,temporada_2022,0.677,0.729,0.702,True,False,True,False
6,d3,2,temporada_2023,0.874,0.854,0.864,True,True,True,True
7,d3,2,temporada_2024,0.748,0.859,0.800,True,True,True,True
8,d3,2,temporada_2026,0.813,0.685,0.744,True,False,True,False
9,d3,3,posterior_2026,0.902,0.713,0.796,True,False,True,False


## 5 · Cifras

In [6]:
def records(df):
    return json.loads(df.to_json(orient="records", date_format="iso", force_ascii=False))

out = {
    "preregistro": "COMBINED_ALERT (commit 4d9d97c)",
    "semanas_brote_2026": outbreaks,
    "veredictos": {k: records(v) for k, v in verdicts.items()},
    "adopta": adopted,
    "lectura_decide_multi_season": {k: records(v) for k, v in reading.items()},
    "antelacion": {k: records(v) for k, v in leads.items()},
    "metricas_por_celda": records(cells),
    "falsas_alarmas_por_celda": {k: records(v) for k, v in added.items()},
    "falsas_alarmas_por_regimen": records(regime),
    "hu0007_4": records(hu),
}
METRICS.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
print(METRICS.name, adopted)

alerta_combinada.json {'d3': False, 'b3': False}
